# 08 Error Analysis and Dissertation Results

This notebook consolidates all completed NER, RE, and end-to-end experiments. It:

- evaluates every NER model against its complete report-level gold entity set;
- reports positive-only RE micro and macro metrics;
- discovers named optimisation experiments without overwriting the baseline;
- keeps literature values separate from internal experimental comparisons.

Run this notebook after each completed Notebook 05, 06, or 07 experiment.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
FIGURES_DIR = RUN_ROOT / "figures"
ANALYSIS_DIR = RUN_ROOT / "analysis"

for path in [RESULTS_DIR, FIGURES_DIR, ANALYSIS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LABEL_MAPS_JSON = INTERIM_DIR / "label_maps.json"
PREP_SUMMARY_JSON = INTERIM_DIR / "prep_summary.json"
ENTITIES_CSV = INTERIM_DIR / "entities.csv"
RELATIONS_CSV = INTERIM_DIR / "relations.csv"
assert LABEL_MAPS_JSON.exists(), "Missing label maps. Run notebook 03 first."
assert ENTITIES_CSV.exists(), "Missing entities.csv. Run notebook 03 first."
assert RELATIONS_CSV.exists(), "Missing relations.csv. Run notebook 03 first."

label_maps = json.loads(LABEL_MAPS_JSON.read_text(encoding="utf-8"))
relation_id_to_label = {
    int(idx): label
    for idx, label in label_maps["relation_id_to_label"].items()
}
relation_labels = [
    relation_id_to_label[idx]
    for idx in sorted(relation_id_to_label)
]
positive_relation_labels = [
    label for label in relation_labels if label != "no_relation"
]

# Results are compared only across experiments that retain this fixed seed.
RANDOM_SEED = 42

# These select detailed error tables and plots. The integrated comparison below
# automatically discovers every completed named experiment.
PRIMARY_NER_EXPERIMENT = "pubmedbert_sliding_window"
PRIMARY_RE_EXPERIMENT = "bert_generic_neg5_unweighted"

NER_METRICS_JSON = RESULTS_DIR / f"ner_{PRIMARY_NER_EXPERIMENT}_full_report_metrics.json"
RE_THRESHOLD_METRICS_JSON = (
    RESULTS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_thresholded_metrics.json"
)
RE_METRICS_JSON = (
    RE_THRESHOLD_METRICS_JSON
    if RE_THRESHOLD_METRICS_JSON.exists()
    else RESULTS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_metrics.json"
)
NER_PREDICTIONS_JSONL = (
    PREDICTIONS_DIR / f"ner_{PRIMARY_NER_EXPERIMENT}_test_predictions.jsonl"
)
RE_PREDICTIONS_CSV = (
    PREDICTIONS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_test_predictions.csv"
)

sns.set_theme(style="whitegrid", context="notebook")

print(
    {
        "project_root": str(PROJECT_ROOT),
        "relation_labels": relation_labels,
        "primary_ner_experiment": PRIMARY_NER_EXPERIMENT,
        "primary_re_experiment": PRIMARY_RE_EXPERIMENT,
    }
)


In [ ]:
def read_json_if_exists(path: Path) -> dict:
    if not path.exists():
        print(f"Missing: {path}")
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl_if_exists(path: Path) -> list[dict]:
    if not path.exists():
        print(f"Missing: {path}")
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            rows.append(json.loads(line))
    return rows


prep_summary = read_json_if_exists(PREP_SUMMARY_JSON)
ner_metrics = read_json_if_exists(NER_METRICS_JSON)
re_metrics = read_json_if_exists(RE_METRICS_JSON)

summary_rows = []
if ner_metrics:
    summary_rows.append(
        {
            "task": "Named entity recognition",
            "experiment": f"ner_{PRIMARY_NER_EXPERIMENT}",
            "precision": ner_metrics.get("precision"),
            "recall": ner_metrics.get("recall"),
            "f1": ner_metrics.get("micro_f1"),
            "accuracy": None,
            "positive_macro_f1": None,
        }
    )
if re_metrics:
    is_thresholded_re = "positive_micro_f1" in re_metrics
    summary_rows.append(
        {
            "task": "Relation extraction",
            "experiment": f"re_{PRIMARY_RE_EXPERIMENT}",
            "precision": (
                re_metrics.get("positive_micro_precision")
                if is_thresholded_re
                else re_metrics.get("test_positive_micro_precision")
            ),
            "recall": (
                re_metrics.get("positive_micro_recall")
                if is_thresholded_re
                else re_metrics.get("test_positive_micro_recall")
            ),
            "f1": (
                re_metrics.get("positive_micro_f1")
                if is_thresholded_re
                else re_metrics.get("test_positive_micro_f1")
            ),
            "accuracy": (
                re_metrics.get("accuracy")
                if is_thresholded_re
                else re_metrics.get("test_accuracy")
            ),
            "positive_macro_f1": (
                re_metrics.get("positive_macro_f1")
                if is_thresholded_re
                else re_metrics.get("test_positive_macro_f1")
            ),
        }
    )

metrics_summary = pd.DataFrame(summary_rows)
metrics_summary_path = RESULTS_DIR / "model_metrics_summary.csv"
metrics_summary.to_csv(metrics_summary_path, index=False)

display(metrics_summary)
print("Saved:", metrics_summary_path)


def to_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().map({"true": True, "false": False}).fillna(False)


In [ ]:
ner_prediction_rows = read_jsonl_if_exists(NER_PREDICTIONS_JSONL)


def entity_set(rows: list[dict], field: str) -> set[tuple]:
    entities = set()
    for row in rows:
        doc_id = row["doc_id"]
        for entity in row[field]:
            entities.add((doc_id, int(entity["start"]), int(entity["end"]), entity["label"]))
    return entities


ner_label_metrics = pd.DataFrame()
if ner_prediction_rows:
    prediction_doc_ids = {row["doc_id"] for row in ner_prediction_rows}
    canonical_entity_frame = pd.read_csv(ENTITIES_CSV)
    canonical_entity_frame = canonical_entity_frame[
        (canonical_entity_frame["split"] == "test")
        & (canonical_entity_frame["doc_id"].isin(prediction_doc_ids))
    ]
    gold_entities = {
        (row.doc_id, int(row.start), int(row.end), row.safe_label)
        for row in canonical_entity_frame.itertuples(index=False)
    }
    predicted_entities = entity_set(ner_prediction_rows, "predicted_entities")

    true_positive = gold_entities.intersection(predicted_entities)
    false_positive = predicted_entities.difference(gold_entities)
    false_negative = gold_entities.difference(predicted_entities)

    labels = sorted({item[3] for item in gold_entities.union(predicted_entities)})
    records = []
    for label in labels:
        gold_label = {item for item in gold_entities if item[3] == label}
        pred_label = {item for item in predicted_entities if item[3] == label}
        tp = len(gold_label.intersection(pred_label))
        fp = len(pred_label.difference(gold_label))
        fn = len(gold_label.difference(pred_label))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        records.append(
            {
                "label": label,
                "gold": len(gold_label),
                "predicted": len(pred_label),
                "true_positive": tp,
                "false_positive": fp,
                "false_negative": fn,
                "precision": precision,
                "recall": recall,
                "f1": f1,
            }
        )

    ner_label_metrics = pd.DataFrame(records).sort_values("label")
    ner_label_metrics_path = RESULTS_DIR / "ner_span_level_metrics_by_label.csv"
    ner_label_metrics.to_csv(ner_label_metrics_path, index=False)

    fp_path = ANALYSIS_DIR / "ner_false_positives.csv"
    fn_path = ANALYSIS_DIR / "ner_false_negatives.csv"
    pd.DataFrame(
        [
            {"doc_id": doc_id, "start": start, "end": end, "label": label, "error_type": "false_positive"}
            for doc_id, start, end, label in sorted(false_positive)
        ]
    ).to_csv(fp_path, index=False)
    pd.DataFrame(
        [
            {"doc_id": doc_id, "start": start, "end": end, "label": label, "error_type": "false_negative"}
            for doc_id, start, end, label in sorted(false_negative)
        ]
    ).to_csv(fn_path, index=False)

    display(ner_label_metrics)
    print("Saved:", ner_label_metrics_path)
    print("Saved:", fp_path)
    print("Saved:", fn_path)
else:
    print(f"NER predictions not found for {PRIMARY_NER_EXPERIMENT}.")

In [ ]:
if not ner_label_metrics.empty:
    plot_data = ner_label_metrics.sort_values("f1", ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(plot_data))))
    sns.barplot(data=plot_data, y="label", x="f1", ax=ax, color="#4C78A8")
    ax.set_title("NER Exact Span F1 by Entity Label")
    ax.set_xlabel("F1")
    ax.set_ylabel("")
    ax.set_xlim(0, 1)
    fig.tight_layout()
    ner_fig_path = FIGURES_DIR / "ner_span_f1_by_label.png"
    fig.savefig(ner_fig_path, dpi=200)
    plt.show()
    print("Saved:", ner_fig_path)

In [ ]:
re_predictions = pd.DataFrame()
if RE_PREDICTIONS_CSV.exists():
    re_predictions = pd.read_csv(RE_PREDICTIONS_CSV)
    y_true = re_predictions["gold_label"]
    y_pred = re_predictions["predicted_label"]

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=relation_labels,
        output_dict=True,
        zero_division=0,
    )
    re_per_label_metrics = (
        pd.DataFrame(report_dict)
        .transpose()
        .reset_index()
        .rename(columns={"index": "label"})
    )
    re_per_label_path = RESULTS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_per_label_metrics.csv"
    re_per_label_metrics.to_csv(re_per_label_path, index=False)

    error_rows = re_predictions[re_predictions["gold_label"] != re_predictions["predicted_label"]].copy()
    error_rows["error_type"] = error_rows["gold_label"] + " -> " + error_rows["predicted_label"]

    re_errors_path = ANALYSIS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_error_examples.csv"
    error_rows.to_csv(re_errors_path, index=False)

    label_error_summary = (
        error_rows.groupby(["gold_label", "predicted_label"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    label_error_path = ANALYSIS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_error_summary_by_label_pair.csv"
    label_error_summary.to_csv(label_error_path, index=False)

    dataset_error_summary = (
        re_predictions.assign(is_error=~to_bool_series(re_predictions["is_correct"]))
        .groupby("dataset")
        .agg(total=("is_error", "size"), errors=("is_error", "sum"))
        .reset_index()
    )
    dataset_error_summary["error_rate"] = dataset_error_summary["errors"] / dataset_error_summary["total"]
    dataset_error_path = ANALYSIS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_error_summary_by_dataset.csv"
    dataset_error_summary.to_csv(dataset_error_path, index=False)

    distance_bins = [-1, 0, 1, 2, 4, 8, 16, 32, np.inf]
    distance_labels = ["0", "1", "2", "3-4", "5-8", "9-16", "17-32", ">32"]
    re_predictions["distance_bin"] = pd.cut(
        re_predictions["distance"],
        bins=distance_bins,
        labels=distance_labels,
    )
    distance_error_summary = (
        re_predictions.assign(is_error=~to_bool_series(re_predictions["is_correct"]))
        .groupby("distance_bin", observed=True)
        .agg(total=("is_error", "size"), errors=("is_error", "sum"))
        .reset_index()
    )
    distance_error_summary["error_rate"] = distance_error_summary["errors"] / distance_error_summary["total"]
    distance_error_path = ANALYSIS_DIR / f"re_{PRIMARY_RE_EXPERIMENT}_error_summary_by_distance.csv"
    distance_error_summary.to_csv(distance_error_path, index=False)

    display(re_per_label_metrics)
    display(label_error_summary.head(20))
    display(dataset_error_summary)
    display(distance_error_summary)
    print("Saved:", re_per_label_path)
    print("Saved:", re_errors_path)
    print("Saved:", label_error_path)
    print("Saved:", dataset_error_path)
    print("Saved:", distance_error_path)
else:
    print("RE predictions not found. Run notebook 05 before this analysis.")

In [ ]:
if not re_predictions.empty:
    cm = confusion_matrix(
        re_predictions["gold_label"],
        re_predictions["predicted_label"],
        labels=relation_labels,
    )
    cm_df = pd.DataFrame(cm, index=relation_labels, columns=relation_labels)

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title("RE Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Gold label")
    fig.tight_layout()
    cm_path = FIGURES_DIR / "re_confusion_matrix.png"
    fig.savefig(cm_path, dpi=200)
    plt.show()

    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    cm_norm_df = pd.DataFrame(cm_norm, index=relation_labels, columns=relation_labels)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm_norm_df, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax)
    ax.set_title("RE Row-Normalised Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Gold label")
    fig.tight_layout()
    cm_norm_path = FIGURES_DIR / "re_confusion_matrix_normalised.png"
    fig.savefig(cm_norm_path, dpi=200)
    plt.show()

    if "distance_bin" in re_predictions.columns:
        distance_plot = (
            re_predictions.assign(is_error=~to_bool_series(re_predictions["is_correct"]))
            .groupby("distance_bin", observed=True)
            .agg(error_rate=("is_error", "mean"), total=("is_error", "size"))
            .reset_index()
        )
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.barplot(data=distance_plot, x="distance_bin", y="error_rate", ax=ax, color="#F58518")
        ax.set_title("RE Error Rate by Entity-Pair Distance")
        ax.set_xlabel("Token distance bin")
        ax.set_ylabel("Error rate")
        ax.set_ylim(0, min(1.0, max(0.05, distance_plot["error_rate"].max() * 1.2)))
        fig.tight_layout()
        distance_fig_path = FIGURES_DIR / "re_error_rate_by_distance.png"
        fig.savefig(distance_fig_path, dpi=200)
        plt.show()

    print("Saved:", cm_path)
    print("Saved:", cm_norm_path)
    if "distance_fig_path" in locals():
        print("Saved:", distance_fig_path)

In [ ]:
report_lines = [
    "# Experiment and Error Analysis Summary",
    "",
    "This file is generated by notebook 08. It summarises model outputs without reproducing raw report text.",
    "",
]

if prep_summary:
    report_lines.extend(
        [
            "## Dataset Preparation",
            "",
            f"- Reports: {prep_summary.get('reports')}",
            f"- Entities: {prep_summary.get('entities')}",
            f"- Relations: {prep_summary.get('relations')}",
            f"- Candidate entity pairs: {prep_summary.get('candidate_pairs')}",
            f"- Positive candidate pairs: {prep_summary.get('positive_candidate_pairs')}",
            f"- Negative candidate pairs: {prep_summary.get('negative_candidate_pairs')}",
            "",
        ]
    )

if not metrics_summary.empty:
    report_lines.extend(["## Model Metrics", "", metrics_summary.to_markdown(index=False), ""])

if not ner_label_metrics.empty:
    report_lines.extend(
        [
            "## NER Notes",
            "",
            "NER is evaluated with exact span and label matching. False positives and false negatives are saved as span-level tables without report text.",
            "",
            f"- Per-label NER metrics: `{(RESULTS_DIR / 'ner_span_level_metrics_by_label.csv').relative_to(PROJECT_ROOT)}`",
            f"- NER F1 figure: `{(FIGURES_DIR / 'ner_span_f1_by_label.png').relative_to(PROJECT_ROOT)}`",
            "",
        ]
    )

if not re_predictions.empty:
    relation_error_rate = 1 - float(to_bool_series(re_predictions["is_correct"]).mean())
    report_lines.extend(
        [
            "## RE Notes",
            "",
            f"- Relation prediction rows analysed: {len(re_predictions)}",
            f"- Overall RE error rate on analysed test candidates: {relation_error_rate:.4f}",
            f"- RE confusion matrix: `{(FIGURES_DIR / 're_confusion_matrix.png').relative_to(PROJECT_ROOT)}`",
            f"- RE normalised confusion matrix: `{(FIGURES_DIR / 're_confusion_matrix_normalised.png').relative_to(PROJECT_ROOT)}`",
            f"- RE error examples: `{(ANALYSIS_DIR / 're_error_examples.csv').relative_to(PROJECT_ROOT)}`",
            "",
            "For the dissertation, prioritise relation-type errors and distance-based errors in the Results and Discussion chapter. Full error tables can be placed in an appendix.",
            "",
        ]
    )

summary_markdown_path = RESULTS_DIR / "experiment_error_analysis_summary.md"
summary_markdown_path.write_text("\n".join(report_lines), encoding="utf-8")

print(summary_markdown_path.read_text(encoding="utf-8"))
print("Saved:", summary_markdown_path)


## Integrated Results

The tables below separate internal experimental results from literature reference values. Literature scores are not treated as directly comparable because both studies use the 2,300-report collection, but the original study used 10-fold cross-validation, report splitting for long sequences, and joint span-based models.


In [ ]:
canonical_entities = pd.read_csv(ENTITIES_CSV)
canonical_entities = canonical_entities[
    canonical_entities["split"] == "test"
].copy()
canonical_relations = pd.read_csv(RELATIONS_CSV)
canonical_relations = canonical_relations[
    canonical_relations["split"] == "test"
].copy()
relation_pair_columns = [
    "doc_id",
    "head_start",
    "head_end",
    "tail_start",
    "tail_end",
]
relation_label_counts = canonical_relations.groupby(
    relation_pair_columns
)["safe_label"].transform("nunique")
canonical_relations = canonical_relations[
    relation_label_counts == 1
].copy()


def entity_set_from_rows(rows: list[dict], field: str) -> set[tuple]:
    return {
        (
            row["doc_id"],
            int(entity["start"]),
            int(entity["end"]),
            entity["label"],
        )
        for row in rows
        for entity in row[field]
    }


def set_scores(gold: set[tuple], predicted: set[tuple]) -> dict:
    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "gold": len(gold),
        "predicted": len(predicted),
    }


def full_report_ner_metrics(rows: list[dict]) -> dict:
    prediction_doc_ids = {row["doc_id"] for row in rows}
    gold_frame = canonical_entities[
        canonical_entities["doc_id"].isin(prediction_doc_ids)
    ]
    gold = {
        (row.doc_id, int(row.start), int(row.end), row.safe_label)
        for row in gold_frame.itertuples(index=False)
    }
    predicted = entity_set_from_rows(rows, "predicted_entities")
    micro = set_scores(gold, predicted)
    labels = sorted({item[-1] for item in gold | predicted})
    per_label = {}
    for label in labels:
        label_gold = {item for item in gold if item[-1] == label}
        label_predicted = {item for item in predicted if item[-1] == label}
        per_label[label] = {
            **set_scores(label_gold, label_predicted),
            "gold": len(label_gold),
            "predicted": len(label_predicted),
        }
    return {
        "precision": micro["precision"],
        "recall": micro["recall"],
        "micro_f1": micro["f1"],
        "macro_f1": float(np.mean([row["f1"] for row in per_label.values()])),
        "gold_entities": len(gold),
        "predicted_entities": len(predicted),
        "per_label": per_label,
    }


ner_comparison_rows = []
for prediction_path in sorted(PREDICTIONS_DIR.glob("ner_*_test_predictions.jsonl")):
    experiment_name = prediction_path.name.removeprefix("ner_").removesuffix(
        "_test_predictions.jsonl"
    )
    rows = read_jsonl_if_exists(prediction_path)
    if not rows:
        continue
    metrics = full_report_ner_metrics(rows)
    config_path = RESULTS_DIR / f"ner_{experiment_name}_run_config.json"
    config = read_json_if_exists(config_path) if config_path.exists() else {}
    ner_comparison_rows.append(
        {
            "experiment": experiment_name,
            "checkpoint": config.get("model_checkpoint"),
            "long_report_method": config.get("long_report_method", "truncation"),
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "micro_f1": metrics["micro_f1"],
            "macro_f1": metrics["macro_f1"],
            "gold_entities": metrics["gold_entities"],
            "random_seed": config.get("random_seed", RANDOM_SEED),
        }
    )
    metrics.update(
        {
            "experiment_name": experiment_name,
            "model_checkpoint": config.get("model_checkpoint"),
            "long_report_method": config.get("long_report_method", "truncation"),
            "random_seed": config.get("random_seed", RANDOM_SEED),
        }
    )
    full_metrics_path = RESULTS_DIR / f"ner_{experiment_name}_full_report_metrics.json"
    full_metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

ner_comparison = pd.DataFrame(ner_comparison_rows).sort_values(
    "micro_f1",
    ascending=False,
)
ner_comparison_path = RESULTS_DIR / "ner_full_report_backbone_comparison.csv"
ner_comparison.to_csv(ner_comparison_path, index=False)

re_comparison_rows = []
for prediction_path in sorted(PREDICTIONS_DIR.glob("re_*_test_predictions.csv")):
    experiment_name = prediction_path.name.removeprefix("re_").removesuffix(
        "_test_predictions.csv"
    )
    frame = pd.read_csv(prediction_path)
    if frame.empty:
        continue
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        frame["gold_label"],
        frame["predicted_label"],
        labels=positive_relation_labels,
        average="micro",
        zero_division=0,
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        frame["gold_label"],
        frame["predicted_label"],
        labels=positive_relation_labels,
        average="macro",
        zero_division=0,
    )
    config_path = RESULTS_DIR / f"re_{experiment_name}_run_config.json"
    config = read_json_if_exists(config_path) if config_path.exists() else {}
    re_comparison_rows.append(
        {
            "experiment": experiment_name,
            "checkpoint": config.get("model_checkpoint"),
            "marker_design": config.get("marker_design", "generic"),
            "negative_ratio": config.get("train_negative_to_positive_ratio"),
            "weighted_loss": config.get("use_class_weighted_loss"),
            "positive_threshold": (
                frame["selected_positive_threshold"].iloc[0]
                if "selected_positive_threshold" in frame.columns
                else None
            ),
            "positive_micro_precision": micro_precision,
            "positive_micro_recall": micro_recall,
            "positive_micro_f1": micro_f1,
            "positive_macro_precision": macro_precision,
            "positive_macro_recall": macro_recall,
            "positive_macro_f1": macro_f1,
            "random_seed": config.get("random_seed", RANDOM_SEED),
        }
    )

re_comparison = pd.DataFrame(re_comparison_rows).sort_values(
    "positive_macro_f1",
    ascending=False,
)
re_comparison_path = RESULTS_DIR / "re_experiment_comparison.csv"
re_comparison.to_csv(re_comparison_path, index=False)

primary_rows = []
for row in ner_comparison.itertuples(index=False):
    primary_rows.append(
        {
            "stage": "NER",
            "experiment": row.experiment,
            "evaluation_scope": "strict full-report entity micro",
            "precision": row.precision,
            "recall": row.recall,
            "f1": row.micro_f1,
            "directly_comparable": True,
        }
    )

for row in re_comparison.itertuples(index=False):
    primary_rows.append(
        {
            "stage": "Gold-entity RE",
            "experiment": row.experiment,
            "evaluation_scope": "positive-relation micro",
            "precision": row.positive_micro_precision,
            "recall": row.positive_micro_recall,
            "f1": row.positive_micro_f1,
            "directly_comparable": True,
        }
    )

def standardise_end_to_end_metrics(
    ner_experiment: str,
    re_experiment: str,
) -> tuple[dict, pd.DataFrame]:
    ner_path = PREDICTIONS_DIR / f"ner_{ner_experiment}_test_predictions.jsonl"
    relation_path = (
        PREDICTIONS_DIR
        / f"end_to_end_{ner_experiment}_{re_experiment}_relations.csv"
    )
    if not ner_path.exists() or not relation_path.exists():
        raise FileNotFoundError(
            f"Missing predictions for {ner_experiment}+{re_experiment}"
        )

    ner_rows = read_jsonl_if_exists(ner_path)
    test_doc_ids = {row["doc_id"] for row in ner_rows}
    predicted_entity_set = entity_set_from_rows(
        ner_rows,
        "predicted_entities",
    )

    entity_frame = canonical_entities[
        canonical_entities["doc_id"].isin(test_doc_ids)
    ]
    relation_frame = canonical_relations[
        canonical_relations["doc_id"].isin(test_doc_ids)
    ]
    predicted_relation_frame = pd.read_csv(relation_path)

    gold_entity_set = {
        (row.doc_id, int(row.start), int(row.end), row.safe_label)
        for row in entity_frame.itertuples(index=False)
    }
    gold_span_relation_set = {
        (
            row.doc_id,
            int(row.head_start),
            int(row.head_end),
            int(row.tail_start),
            int(row.tail_end),
            row.safe_label,
        )
        for row in relation_frame.itertuples(index=False)
    }
    predicted_span_relation_set = {
        (
            row.doc_id,
            int(row.head_start),
            int(row.head_end),
            int(row.tail_start),
            int(row.tail_end),
            row.predicted_label,
        )
        for row in predicted_relation_frame.itertuples(index=False)
    }

    gold_entity_label_lookup = {
        (row.doc_id, int(row.start), int(row.end)): row.safe_label
        for row in entity_frame.itertuples(index=False)
    }
    gold_graph_relation_set = {
        (
            row.doc_id,
            int(row.head_start),
            int(row.head_end),
            gold_entity_label_lookup.get(
                (row.doc_id, int(row.head_start), int(row.head_end)),
                "<missing>",
            ),
            int(row.tail_start),
            int(row.tail_end),
            gold_entity_label_lookup.get(
                (row.doc_id, int(row.tail_start), int(row.tail_end)),
                "<missing>",
            ),
            row.safe_label,
        )
        for row in relation_frame.itertuples(index=False)
    }
    predicted_graph_relation_set = {
        (
            row.doc_id,
            int(row.head_start),
            int(row.head_end),
            row.head_label,
            int(row.tail_start),
            int(row.tail_end),
            row.tail_label,
            row.predicted_label,
        )
        for row in predicted_relation_frame.itertuples(index=False)
    }

    standardised = {
        "ner_experiment": ner_experiment,
        "re_experiment": re_experiment,
        "random_seed": RANDOM_SEED,
        "gold_source": "canonical entities.csv and relations.csv",
        "ambiguous_pair_rule": "exclude ordered pairs with multiple labels",
        "ner_strict_entity": set_scores(
            gold_entity_set,
            predicted_entity_set,
        ),
        "end_to_end_relation_span_and_label": set_scores(
            gold_span_relation_set,
            predicted_span_relation_set,
        ),
        "end_to_end_graph_entity_and_relation_labels": set_scores(
            gold_graph_relation_set,
            predicted_graph_relation_set,
        ),
    }

    per_relation_rows = []
    relation_names = sorted(
        {item[-1] for item in gold_span_relation_set}
        | {item[-1] for item in predicted_span_relation_set}
    )
    for relation_label in relation_names:
        gold_subset = {
            item
            for item in gold_span_relation_set
            if item[-1] == relation_label
        }
        predicted_subset = {
            item
            for item in predicted_span_relation_set
            if item[-1] == relation_label
        }
        per_relation_rows.append(
            {
                "relation_label": relation_label,
                **set_scores(gold_subset, predicted_subset),
            }
        )
    return standardised, pd.DataFrame(per_relation_rows)


end_to_end_runs = {}
for metrics_path in sorted(RESULTS_DIR.glob("end_to_end_*_metrics.json")):
    if metrics_path.name.endswith("_standardized_metrics.json"):
        continue
    run_metrics = read_json_if_exists(metrics_path)
    if not run_metrics:
        continue
    ner_experiment = run_metrics["ner_experiment"]
    re_experiment = run_metrics["re_experiment"]
    run_key = f"{ner_experiment}+{re_experiment}"
    standardised, standardised_per_relation = standardise_end_to_end_metrics(
        ner_experiment,
        re_experiment,
    )
    end_to_end_runs[run_key] = standardised

    standardised_path = (
        RESULTS_DIR
        / f"end_to_end_{ner_experiment}_{re_experiment}_standardized_metrics.json"
    )
    standardised_path.write_text(
        json.dumps(standardised, indent=2),
        encoding="utf-8",
    )
    standardised_relation_path = (
        RESULTS_DIR
        / f"end_to_end_{ner_experiment}_{re_experiment}_standardized_per_relation.csv"
    )
    standardised_per_relation.to_csv(
        standardised_relation_path,
        index=False,
    )

    for metric_key, scope in [
        ("ner_strict_entity", "strict full-report entity micro"),
        ("end_to_end_relation_span_and_label", "strict relation span+label micro"),
        (
            "end_to_end_graph_entity_and_relation_labels",
            "strict graph micro",
        ),
    ]:
        values = standardised[metric_key]
        primary_rows.append(
            {
                "stage": "End-to-end",
                "experiment": run_key,
                "evaluation_scope": scope,
                "precision": values["precision"],
                "recall": values["recall"],
                "f1": values["f1"],
                "directly_comparable": True,
            }
        )

primary_results = pd.DataFrame(primary_rows)
primary_results_path = RESULTS_DIR / "dissertation_primary_results.csv"
primary_results.to_csv(primary_results_path, index=False)

literature_reference = pd.DataFrame(
    [
        {
            "framework": "SpERT",
            "backbone": "BERT",
            "entity_micro_f1": 0.844,
            "relation_micro_f1": 0.638,
        },
        {
            "framework": "DyGIE++",
            "backbone": "BERT",
            "entity_micro_f1": 0.877,
            "relation_micro_f1": 0.729,
        },
        {
            "framework": "DyGIE++",
            "backbone": "BiomedBERT",
            "entity_micro_f1": 0.880,
            "relation_micro_f1": 0.725,
        },
        {
            "framework": "DyGIE++",
            "backbone": "BiomedVLP-CXR-BERT",
            "entity_micro_f1": 0.889,
            "relation_micro_f1": 0.737,
        },
    ]
)
literature_reference["dataset_protocol"] = (
    "full RadGraph-XL; 10-fold CV; joint extraction"
)
literature_reference["directly_comparable"] = False
literature_path = RESULTS_DIR / "literature_reference_results.csv"
literature_reference.to_csv(literature_path, index=False)

display(ner_comparison)
display(re_comparison)
display(primary_results)
display(literature_reference)

print("Saved:", ner_comparison_path)
print("Saved:", re_comparison_path)
print("Saved:", primary_results_path)
print("Saved:", literature_path)


In [ ]:
relation_comparison_rows = []

for prediction_path in sorted(PREDICTIONS_DIR.glob("re_*_test_predictions.csv")):
    experiment_name = prediction_path.name.removeprefix("re_").removesuffix(
        "_test_predictions.csv"
    )
    frame = pd.read_csv(prediction_path)
    report = classification_report(
        frame["gold_label"],
        frame["predicted_label"],
        labels=positive_relation_labels,
        output_dict=True,
        zero_division=0,
    )
    for label in positive_relation_labels:
        values = report[label]
        relation_comparison_rows.append(
            {
                "setting": "gold entities",
                "experiment": experiment_name,
                "relation_label": label,
                "precision": values["precision"],
                "recall": values["recall"],
                "f1": values["f1-score"],
                "support": values["support"],
            }
        )

for run_key, run_metrics in end_to_end_runs.items():
    ner_experiment = run_metrics["ner_experiment"]
    re_experiment = run_metrics["re_experiment"]
    path = RESULTS_DIR / f"end_to_end_{ner_experiment}_{re_experiment}_standardized_per_relation.csv"
    if not path.exists():
        continue
    frame = pd.read_csv(path)
    for row in frame.itertuples(index=False):
        relation_comparison_rows.append(
            {
                "setting": "end-to-end",
                "experiment": run_key,
                "relation_label": row.relation_label,
                "precision": row.precision,
                "recall": row.recall,
                "f1": row.f1,
                "support": row.gold,
            }
        )

relation_comparison = pd.DataFrame(relation_comparison_rows)
relation_comparison_path = RESULTS_DIR / "dissertation_relation_results.csv"
relation_comparison.to_csv(relation_comparison_path, index=False)
display(relation_comparison)

if not primary_results.empty:
    plot_rows = primary_results[
        ~primary_results["evaluation_scope"].str.contains(
            "full-report entity",
            regex=False,
        )
    ].copy()
    fig, ax = plt.subplots(figsize=(10, max(4, 0.55 * len(plot_rows))))
    labels = (
        plot_rows["stage"]
        + ": "
        + plot_rows["experiment"]
        + "\n"
        + plot_rows["evaluation_scope"]
    )
    ax.barh(labels, plot_rows["f1"], color="#356859")
    ax.set_xlim(0, 1)
    ax.set_xlabel("F1")
    ax.set_ylabel("")
    ax.set_title("Primary Pipeline Results")
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()
    primary_figure_path = FIGURES_DIR / "dissertation_primary_results_f1.png"
    fig.savefig(primary_figure_path, dpi=200, bbox_inches="tight")
    plt.show()

if not relation_comparison.empty:
    selected_for_plot = relation_comparison[
        relation_comparison["experiment"].isin(
            [
                PRIMARY_RE_EXPERIMENT,
                f"{PRIMARY_NER_EXPERIMENT}+{PRIMARY_RE_EXPERIMENT}",
            ]
        )
    ].copy()
    if not selected_for_plot.empty:
        fig, ax = plt.subplots(figsize=(9, 5))
        sns.barplot(
            data=selected_for_plot,
            x="relation_label",
            y="f1",
            hue="setting",
            ax=ax,
        )
        ax.set_ylim(0, 1)
        ax.set_xlabel("")
        ax.set_ylabel("F1")
        ax.set_title("Relation F1: Gold Entities vs End-to-End")
        ax.legend(title="")
        fig.tight_layout()
        relation_figure_path = FIGURES_DIR / "dissertation_relation_f1_comparison.png"
        fig.savefig(relation_figure_path, dpi=200, bbox_inches="tight")
        plt.show()

summary_lines = [
    "# Dissertation Results and Error Analysis",
    "",
    "## Full-Report NER Comparison",
    "",
    ner_comparison.to_markdown(index=False)
    if not ner_comparison.empty
    else "No NER experiments are available.",
    "",
    "## Gold-Entity RE Comparison",
    "",
    re_comparison.to_markdown(index=False)
    if not re_comparison.empty
    else "No RE experiments are available.",
    "",
    "## End-to-End and Primary Results",
    "",
    primary_results.to_markdown(index=False),
    "",
    "## Literature Context",
    "",
    literature_reference.to_markdown(index=False),
    "",
    (
        "The literature values are contextual rather than directly comparable because "
        "the split protocol, architecture, and evaluation setting differ."
    ),
    "",
    "## Interpretation Checklist",
    "",
    "- Use strict full-report NER metrics, not tokenizer-specific truncated support.",
    "- Select RE thresholds on validation data only.",
    "- Prioritise positive micro and macro F1 over candidate accuracy.",
    "- Keep gold-entity RE separate from end-to-end RE.",
    "- Compare experiments only when one controlled factor changes.",
    "- All current optimisation profiles retain random seed 42.",
]
final_summary_path = RESULTS_DIR / "dissertation_results_and_error_analysis.md"
final_summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print("Saved:", relation_comparison_path)
print("Saved:", final_summary_path)
